# RNN for Turbofan Engine RUL Prediction

## Overview

This educational notebook demonstrates how to use a **Recurrent Neural Network (RNN)** for time series prediction of Remaining Useful Life (RUL) of turbofan engines.

## Learning Objectives

By the end of this notebook, you will:
- Understand how RNNs process sequential data
- Learn to create time series sequences from engine data
- Build and train an RNN model for RUL prediction
- Evaluate RNN performance and interpret results

## What is an RNN?

**Recurrent Neural Networks (RNNs)** are designed to process sequences of data by maintaining a "memory" of previous inputs. Unlike feedforward networks, RNNs have connections that form cycles, allowing information to persist.

### Key Concepts:
- **Sequential Processing**: Processes data one timestep at a time
- **Hidden State**: Maintains information about previous inputs
- **Time Dependencies**: Can learn patterns that depend on sequence order

### Advantages:
- Naturally handles sequential data
- Can learn temporal patterns
- Memory of previous states

### Limitations:
- Can struggle with long sequences (vanishing gradient problem)
- Training can be slower than feedforward networks


## ⚠️ IMPORTANT: GPU/CUDA Error Fix

**If you encounter GPU/CUDA errors**, please:

1. **Restart the kernel**: Kernel → Restart Kernel (or Restart & Clear Output)
2. **Run the import cell below FIRST** - it will disable GPU and use CPU only
3. **Then run the rest of the cells**

The GPU disabling code must run BEFORE TensorFlow is imported. If TensorFlow was already loaded, restarting the kernel is required.


## 1. Import Libraries

### Purpose
Import deep learning libraries (TensorFlow/Keras) and data processing tools needed for RNN implementation.

### Key Libraries:
- **TensorFlow/Keras**: Deep learning framework for building RNN models
- **SimpleRNN**: Basic RNN layer implementation
- **MinMaxScaler**: Normalizes data to [0,1] range (important for neural networks)

### Note:
This cell disables GPU to avoid CUDA/libdevice errors. Models will run on CPU (slower but more reliable).


In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import time  # For measuring training time
warnings.filterwarnings('ignore')

# Configure TensorFlow to use CPU only (avoids CUDA/libdevice issues)
# IMPORTANT: Set this BEFORE importing TensorFlow for it to take effect
# If you still see GPU errors, restart the kernel and run this cell first!
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # Disable GPU, use CPU only
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # Suppress TensorFlow warnings

# Deep Learning
import tensorflow as tf

# Aggressively disable GPU and force CPU usage
try:
    # Hide all GPU devices
    tf.config.set_visible_devices([], 'GPU')
    # Set memory growth to prevent GPU allocation
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, False)
except:
    pass

# Force CPU device placement for all operations
tf.config.set_soft_device_placement(True)
with tf.device('/CPU:0'):
    # This ensures CPU is used by default
    pass

from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Set style
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    try:
        plt.style.use('seaborn-darkgrid')
    except:
        plt.style.use('ggplot')
sns.set_palette("husl")
%matplotlib inline

print("Libraries imported successfully!")
print(f"TensorFlow version: {tf.__version__}")

# Verify GPU is disabled
print(f"\n📊 TensorFlow Device Configuration:")
print(f"   Available GPUs: {len(tf.config.list_physical_devices('GPU'))}")
print(f"   Available CPUs: {len(tf.config.list_physical_devices('CPU'))}")
if len(tf.config.list_physical_devices('GPU')) == 0:
    print("   ✅ GPU successfully disabled - using CPU only")
else:
    print("   ⚠️  WARNING: GPU still detected! Please restart kernel and run this cell first.")


2026-01-17 22:35:42.542426: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Libraries imported successfully!
TensorFlow version: 2.20.0


## 2. Load and Prepare Data

### Purpose
Load the turbofan engine dataset. The data will be processed into sequences in the next step.

### Note:
For time series models, we need to structure the data differently than traditional ML models - we'll create sequences of consecutive timesteps.


In [2]:
# Define data path
data_path = Path('../../dataset/6.+Turbofan+Engine+Degradation+Simulation+Data+Set/6. Turbofan Engine Degradation Simulation Data Set/CMAPSSData')

# Column names with actual field names
op_settings = ['Altitude', 'Mach', 'TRA']
sensors = ['T2', 'T24', 'T30', 'T50', 'P2', 'P15', 'P30', 'Nf', 'Nc', 'epr', 'Ps30', 'phi', 
           'NRf', 'NRc', 'BPR', 'farB', 'htBleed', 'Nf_dmd', 'PCNfR_dmd', 'W31', 'W32']
column_names = ['unit', 'time'] + op_settings + sensors

def load_data(dataset='FD001'):
    """Load training and test data"""
    train_file = data_path / f'train_{dataset}.csv'
    test_file = data_path / f'test_{dataset}.csv'
    rul_file = data_path / f'RUL_{dataset}.csv'
    
    # CSV files already have headers, so no need for sep, header=None, or names
    train_df = pd.read_csv(train_file)
    test_df = pd.read_csv(test_file)
    rul_df = pd.read_csv(rul_file)
    
    return train_df, test_df, rul_df

def calculate_rul_train(df):
    """Calculate RUL for training data"""
    df = df.copy()
    df['RUL'] = df.groupby('unit')['time'].transform(lambda x: x.max() - x)
    return df

# Load data
train_df, test_df, rul_df = load_data('FD001')
train_df = calculate_rul_train(train_df)

print(f"Training data shape: {train_df.shape}")
print(f"Test data shape: {test_df.shape}")
print(f"Number of engines in training: {train_df['unit'].nunique()}")
print(f"Number of engines in test: {test_df['unit'].nunique()}")


: 

## 3. Create Time Series Sequences

### Purpose
Transform the data into sequences that RNNs can process. This is a critical step for time series modeling.

### What are Sequences?
Instead of using individual rows as samples, we create **sliding windows** of consecutive timesteps:
- **Input**: Last N timesteps (e.g., 30 cycles) of sensor readings
- **Output**: RUL at the end of that sequence

### Example:
If sequence_length = 30:
- Sequence 1: Cycles 1-30 → Predict RUL at cycle 30
- Sequence 2: Cycles 2-31 → Predict RUL at cycle 31
- Sequence 3: Cycles 3-32 → Predict RUL at cycle 32
- etc.

### Why This Matters:
- RNNs learn from the **pattern** of sensor changes over time
- Captures temporal dependencies (how current state depends on previous states)
- More informative than using single timestep features


In [ ]:
def create_sequences(data, sequence_length=30):
    """Create sequences for time series prediction"""
    sequences = []
    targets = []
    
    # Select features (using actual field names)
    op_settings = ['Altitude', 'Mach', 'TRA']
    sensors = ['T2', 'T24', 'T30', 'T50', 'P2', 'P15', 'P30', 'Nf', 'Nc', 'epr', 'Ps30', 'phi', 
               'NRf', 'NRc', 'BPR', 'farB', 'htBleed', 'Nf_dmd', 'PCNfR_dmd', 'W31', 'W32']
    feature_cols = op_settings + sensors
    
    for unit_id in data['unit'].unique():
        unit_data = data[data['unit'] == unit_id].sort_values('time')
        unit_features = unit_data[feature_cols].values
        unit_rul = unit_data['RUL'].values
        
        # Create sequences
        for i in range(len(unit_data) - sequence_length + 1):
            sequences.append(unit_features[i:i+sequence_length])
            targets.append(unit_rul[i+sequence_length-1])
    
    return np.array(sequences), np.array(targets)

# Create sequences
sequence_length = 30
X_train_seq, y_train_seq = create_sequences(train_df, sequence_length)

# For test data, we need to get the last sequence_length timesteps for each engine
def create_test_sequences(data, sequence_length=30):
    """Create test sequences (last sequence_length timesteps for each engine)"""
    sequences = []
    op_settings = ['Altitude', 'Mach', 'TRA']
    sensors = ['T2', 'T24', 'T30', 'T50', 'P2', 'P15', 'P30', 'Nf', 'Nc', 'epr', 'Ps30', 'phi', 
               'NRf', 'NRc', 'BPR', 'farB', 'htBleed', 'Nf_dmd', 'PCNfR_dmd', 'W31', 'W32']
    feature_cols = op_settings + sensors
    
    for unit_id in sorted(data['unit'].unique()):
        unit_data = data[data['unit'] == unit_id].sort_values('time')
        unit_features = unit_data[feature_cols].values
        
        # Take last sequence_length timesteps
        if len(unit_features) >= sequence_length:
            sequences.append(unit_features[-sequence_length:])
        else:
            # Pad if shorter
            padding = np.zeros((sequence_length - len(unit_features), len(feature_cols)))
            sequences.append(np.vstack([padding, unit_features]))
    
    return np.array(sequences)

X_test_seq = create_test_sequences(test_df, sequence_length)
y_test = rul_df['RUL'].values

print(f"Training sequences shape: {X_train_seq.shape}")
print(f"Training targets shape: {y_train_seq.shape}")
print(f"Test sequences shape: {X_test_seq.shape}")
print(f"Test targets shape: {y_test.shape}")


## 4. Data Normalization

### Purpose
Normalize features and targets to [0,1] range using MinMaxScaler. This is essential for neural networks because:
- Neural networks are sensitive to input scale
- Normalization helps with gradient stability during training
- Prevents features with large values from dominating

### Why MinMaxScaler (not StandardScaler)?
- Neural networks often work better with [0,1] range
- Ensures all features are on the same scale
- Helps with activation function behavior (especially sigmoid/tanh)

### Important:
We normalize targets (RUL) too, then inverse-transform predictions back to original scale for evaluation.


In [ ]:
# Normalize features
feature_scaler = MinMaxScaler()
n_samples, n_timesteps, n_features = X_train_seq.shape
X_train_reshaped = X_train_seq.reshape(-1, n_features)
X_train_scaled = feature_scaler.fit_transform(X_train_reshaped)
X_train_seq_scaled = X_train_scaled.reshape(n_samples, n_timesteps, n_features)

# Normalize test data
X_test_reshaped = X_test_seq.reshape(-1, n_features)
X_test_scaled = feature_scaler.transform(X_test_reshaped)
X_test_seq_scaled = X_test_scaled.reshape(X_test_seq.shape)

# Normalize targets
target_scaler = MinMaxScaler()
y_train_scaled = target_scaler.fit_transform(y_train_seq.reshape(-1, 1)).flatten()
y_test_scaled = target_scaler.transform(y_test.reshape(-1, 1)).flatten()

print("Data normalized successfully!")
print(f"Training features - Min: {X_train_seq_scaled.min():.4f}, Max: {X_train_seq_scaled.max():.4f}")
print(f"Training targets - Min: {y_train_scaled.min():.4f}, Max: {y_train_scaled.max():.4f}")


## 5. Build RNN Model

### Purpose
Construct the RNN architecture. Our model uses:
- **Two RNN layers**: First processes sequences, second extracts final representation
- **Dropout layers**: Prevent overfitting by randomly setting some neurons to zero
- **Dense layers**: Final layers that map RNN output to RUL prediction

### Architecture:
1. First RNN layer (64 units, returns sequences): Processes each timestep
2. Dropout (20%): Regularization
3. Second RNN layer (32 units, returns single value): Extracts final representation
4. Dropout (20%): More regularization
5. Dense layer (16 units): Feature transformation
6. Output layer (1 unit): Final RUL prediction

### Model Parameters:
- **Units**: Number of neurons in each layer (more = more capacity, but slower)
- **Activation**: 'relu' for hidden layers (helps with gradient flow)
- **Dropout rate**: 0.2 means 20% of neurons are randomly disabled during training


In [ ]:
def build_rnn_model(sequence_length, n_features):
    """Build RNN model"""
    # Force CPU device placement to avoid GPU errors
    with tf.device('/CPU:0'):
        model = Sequential([
            SimpleRNN(64, activation='relu', return_sequences=True, input_shape=(sequence_length, n_features)),
            Dropout(0.2),
            SimpleRNN(32, activation='relu', return_sequences=False),
            Dropout(0.2),
            Dense(16, activation='relu'),
            Dense(1)
        ])
        
        model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])
    return model

# Build model
n_features = X_train_seq_scaled.shape[2]
model = build_rnn_model(sequence_length, n_features)
model.summary()


## 6. Train Model

### Purpose
Train the RNN model on the sequence data. Training involves:
- Forward pass: Model makes predictions
- Loss calculation: Compare predictions to actual RUL
- Backpropagation: Update weights to reduce error
- Repeat for multiple epochs

### Training Parameters:
- **Epochs**: Number of complete passes through the training data
- **Batch size**: Number of samples processed before updating weights (32 = process 32 sequences at once)
- **Validation split**: 20% of data held out for validation (not used for training)

### Callbacks:
- **Early Stopping**: Stops training if validation loss doesn't improve (prevents overfitting)
- **Reduce Learning Rate**: Automatically reduces learning rate if stuck (helps find better solutions)


In [ ]:
# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

# Train model
history = model.fit(
    X_train_seq_scaled, y_train_scaled,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)


## 7. Evaluate Model

### Purpose
Evaluate the trained RNN model's performance on both training and test sets. This tells us:
- How well the model learned the training patterns
- How well it generalizes to unseen data
- Whether the model is overfitting (large train-test gap)

### Metrics Explained:
- **RMSE**: Root Mean Squared Error - penalizes large errors more
- **MAE**: Mean Absolute Error - average prediction error
- **R²**: Coefficient of determination - proportion of variance explained

### What Good Performance Looks Like:
- Low RMSE and MAE (close predictions)
- High R² (close to 1.0)
- Similar train and test performance (not overfitting)


## 7.5. Parameter Tuning with K-Fold Cross-Validation

### Purpose
Improve model performance by finding the best hyperparameters using cross-validation.

### What is K-Fold Cross-Validation?

**K-Fold Cross-Validation** is a technique to evaluate model performance more reliably:

1. **Split data into K folds** (e.g., K=5 means 5 groups)
2. **Train on K-1 folds**, test on 1 fold
3. **Repeat K times**, each time using a different fold as test set
4. **Average the results** across all K folds

### Why Use Cross-Validation?

- **More reliable**: Tests on multiple different data splits
- **Better hyperparameter selection**: Finds parameters that work well across different data
- **Reduces overfitting risk**: Model is tested on data it hasn't seen during training
- **Better generalization estimate**: More accurate prediction of real-world performance

### Time Series Cross-Validation

For time series data, we use **TimeSeriesSplit** which:
- Respects temporal order (doesn't shuffle)
- Uses past data to predict future
- Prevents data leakage (no future data in training)

### Hyperparameters We'll Tune:

- **Units in RNN layers**: Number of neurons (32, 64, 128)
- **Learning rate**: How fast the model learns (0.001, 0.0001)
- **Batch size**: Samples per update (16, 32, 64)
- **Dropout rate**: Regularization strength (0.1, 0.2, 0.3)

In [ ]:
# Import additional libraries for cross-validation and parameter tuning
from sklearn.model_selection import TimeSeriesSplit, KFold
from scipy.stats import uniform, randint
import itertools

print("✅ Cross-validation libraries imported!")

### Understanding K-Fold Cross-Validation

Let's first understand how K-Fold works with a simple example.

In [ ]:
# Demonstrate K-Fold Cross-Validation concept
from sklearn.model_selection import KFold

# Simple example: 10 data points, 5 folds
data_size = 10
k_folds = 5
kf = KFold(n_splits=k_folds, shuffle=False)

print("📊 K-Fold Cross-Validation Example (K=5):")
print("=" * 60)
print(f"Total data points: {data_size}")
print(f"Number of folds: {k_folds}")
print(f"\nFold splits:")

for fold, (train_idx, val_idx) in enumerate(kf.split(range(data_size)), 1):
    print(f"\nFold {fold}:")
    print(f"  Training indices: {train_idx}")
    print(f"  Validation indices: {val_idx}")
    print(f"  Train size: {len(train_idx)}, Val size: {len(val_idx)}")

print("\n💡 Key Points:")
print("   - Each data point is used for validation exactly once")
print("   - Model is trained and tested K times")
print("   - Final score = average of all K test scores")

### Time Series Cross-Validation

For time series data, we use **TimeSeriesSplit** which respects temporal order.

In [ ]:
# Demonstrate Time Series Cross-Validation
from sklearn.model_selection import TimeSeriesSplit

# Example with time series data
tscv = TimeSeriesSplit(n_splits=5)

print("📊 Time Series Cross-Validation Example (5 splits):")
print("=" * 60)
print("Key difference: Respects temporal order (no shuffling)")
print("\nFold splits:")

for fold, (train_idx, val_idx) in enumerate(tscv.split(range(20)), 1):
    print(f"\nFold {fold}:")
    print(f"  Training: indices 0 to {train_idx[-1]} (size: {len(train_idx)})")
    print(f"  Validation: indices {val_idx[0]} to {val_idx[-1]} (size: {len(val_idx)})")
    print(f"  ⚠️  Validation always comes AFTER training (temporal order preserved)")

print("\n💡 Why this matters for time series:")
print("   - We predict future based on past")
print("   - No data leakage from future to past")
print("   - More realistic evaluation")

### Manual Hyperparameter Search with Cross-Validation

Let's create a function to evaluate different hyperparameter combinations using cross-validation.

In [ ]:
def evaluate_hyperparameters(X, y, param_combinations, n_splits=3, epochs=20, verbose=0):
    """
    Evaluate different hyperparameter combinations using K-Fold Cross-Validation
    
    Parameters:
    -----------
    X : array-like
        Training features
    y : array-like
        Training targets
    param_combinations : list of dict
        List of parameter dictionaries to try
    n_splits : int
        Number of folds for cross-validation
    epochs : int
        Number of training epochs
    verbose : int
        Verbosity level (0 = silent, 1 = progress)
    
    Returns:
    --------
    results : list of dict
        Results for each parameter combination
    """
    from sklearn.model_selection import KFold
    from sklearn.metrics import mean_squared_error, r2_score
    
    results = []
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    print(f"🔍 Evaluating {len(param_combinations)} parameter combinations...")
    print(f"   Using {n_splits}-fold cross-validation")
    print("=" * 70)
    
    for idx, params in enumerate(param_combinations, 1):
        print(f"\n[{idx}/{len(param_combinations)}] Testing: {params}")
        
        fold_scores = []
        fold_r2_scores = []
        
        for fold, (train_idx, val_idx) in enumerate(kf.split(X), 1):
            if verbose > 0:
                print(f"  Fold {fold}/{n_splits}...", end=" ")
            
            # Split data for this fold
            X_train_fold = X[train_idx]
            y_train_fold = y[train_idx]
            X_val_fold = X[val_idx]
            y_val_fold = y[val_idx]
            
            # Build model with current parameters
            with tf.device('/CPU:0'):
                model = Sequential([
                    SimpleRNN(params['units1'], activation='relu', 
                             return_sequences=True, input_shape=(X.shape[1], X.shape[2])),
                    Dropout(params['dropout']),
                    SimpleRNN(params['units2'], activation='relu', return_sequences=False),
                    Dropout(params['dropout']),
                    Dense(16, activation='relu'),
                    Dense(1)
                ])
                
                model.compile(
                    optimizer=Adam(learning_rate=params['learning_rate']), 
                    loss='mse', 
                    metrics=['mae']
                )
            
            # Train model
            model.fit(
                X_train_fold, y_train_fold,
                epochs=epochs,
                batch_size=params['batch_size'],
                validation_data=(X_val_fold, y_val_fold),
                verbose=0,
                callbacks=[EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)]
            )
            
            # Evaluate on validation set
            y_pred_fold = model.predict(X_val_fold, verbose=0)
            rmse_fold = np.sqrt(mean_squared_error(y_val_fold, y_pred_fold))
            r2_fold = r2_score(y_val_fold, y_pred_fold)
            
            fold_scores.append(rmse_fold)
            fold_r2_scores.append(r2_fold)
            
            if verbose > 0:
                print(f"RMSE: {rmse_fold:.4f}, R²: {r2_fold:.4f}")
        
        # Calculate average scores across folds
        avg_rmse = np.mean(fold_scores)
        std_rmse = np.std(fold_scores)
        avg_r2 = np.mean(fold_r2_scores)
        std_r2 = np.std(fold_r2_scores)
        
        results.append({
            'params': params,
            'avg_rmse': avg_rmse,
            'std_rmse': std_rmse,
            'avg_r2': avg_r2,
            'std_r2': std_r2,
            'fold_scores': fold_scores
        })
        
        print(f"  ✅ Average RMSE: {avg_rmse:.4f} (±{std_rmse:.4f})")
        print(f"     Average R²: {avg_r2:.4f} (±{std_r2:.4f})")
    
    return results

print("✅ Hyperparameter evaluation function created!")

### Define Hyperparameter Search Space

Let's define different combinations of hyperparameters to test.

In [ ]:
# Define hyperparameter search space
# We'll test different combinations of key hyperparameters

param_combinations = [
    # Combination 1: Small model, default learning rate
    {
        'units1': 32,
        'units2': 16,
        'learning_rate': 0.001,
        'batch_size': 32,
        'dropout': 0.2
    },
    # Combination 2: Medium model, default learning rate
    {
        'units1': 64,
        'units2': 32,
        'learning_rate': 0.001,
        'batch_size': 32,
        'dropout': 0.2
    },
    # Combination 3: Large model, default learning rate
    {
        'units1': 128,
        'units2': 64,
        'learning_rate': 0.001,
        'batch_size': 32,
        'dropout': 0.2
    },
    # Combination 4: Medium model, lower learning rate
    {
        'units1': 64,
        'units2': 32,
        'learning_rate': 0.0001,
        'batch_size': 32,
        'dropout': 0.2
    },
    # Combination 5: Medium model, higher dropout
    {
        'units1': 64,
        'units2': 32,
        'learning_rate': 0.001,
        'batch_size': 32,
        'dropout': 0.3
    },
    # Combination 6: Medium model, smaller batch size
    {
        'units1': 64,
        'units2': 32,
        'learning_rate': 0.001,
        'batch_size': 16,
        'dropout': 0.2
    },
]

print(f"📋 Defined {len(param_combinations)} hyperparameter combinations to test:")
print("=" * 70)
for i, params in enumerate(param_combinations, 1):
    print(f"\n{i}. Units: [{params['units1']}, {params['units2']}], "
          f"LR: {params['learning_rate']}, "
          f"Batch: {params['batch_size']}, "
          f"Dropout: {params['dropout']}")

### Run Cross-Validation for Hyperparameter Tuning

**Note**: This will take some time as we train multiple models. We'll use 3-fold CV with fewer epochs for speed.

In [ ]:
# Run hyperparameter search with cross-validation
# Note: This takes time! Using 3-fold CV and 20 epochs for speed
# For better results, increase n_splits and epochs

print("🚀 Starting Hyperparameter Tuning with Cross-Validation...")
print("⚠️  This may take 10-20 minutes depending on your system...")
print("=" * 70)

start_time = time.time()

# Use a sample of data for faster tuning (optional - remove for full dataset)
# For demonstration, we'll use all data but with fewer epochs
tuning_results = evaluate_hyperparameters(
    X_train_seq_scaled, 
    y_train_scaled,
    param_combinations,
    n_splits=3,  # 3-fold cross-validation
    epochs=20,  # Fewer epochs for speed
    verbose=1
)

tuning_time = time.time() - start_time

print(f"\n✅ Hyperparameter tuning completed in {tuning_time/60:.1f} minutes!")

### Analyze Cross-Validation Results

Let's compare the results and find the best hyperparameters.

In [ ]:
# Create results DataFrame for easy comparison
results_df = pd.DataFrame([
    {
        'Units1': r['params']['units1'],
        'Units2': r['params']['units2'],
        'Learning Rate': r['params']['learning_rate'],
        'Batch Size': r['params']['batch_size'],
        'Dropout': r['params']['dropout'],
        'Avg RMSE': r['avg_rmse'],
        'Std RMSE': r['std_rmse'],
        'Avg R²': r['avg_r2'],
        'Std R²': r['std_r2']
    }
    for r in tuning_results
])

# Sort by best R² score
results_df = results_df.sort_values('Avg R²', ascending=False)

print("📊 Cross-Validation Results (Sorted by R² Score):")
print("=" * 100)
print(results_df.to_string(index=False))

# Find best parameters
best_result = max(tuning_results, key=lambda x: x['avg_r2'])
best_params = best_result['params']

print(f"\n🏆 Best Hyperparameters (by Cross-Validation R²):")
print("=" * 70)
print(f"   Units: [{best_params['units1']}, {best_params['units2']}]")
print(f"   Learning Rate: {best_params['learning_rate']}")
print(f"   Batch Size: {best_params['batch_size']}")
print(f"   Dropout: {best_params['dropout']}")
print(f"\n📈 Best Cross-Validation Performance:")
print(f"   Average RMSE: {best_result['avg_rmse']:.4f} (±{best_result['std_rmse']:.4f})")
print(f"   Average R²: {best_result['avg_r2']:.4f} (±{best_result['std_r2']:.4f})")

### Visualize Cross-Validation Results

Let's create visualizations to compare different hyperparameter combinations.

In [ ]:
# Visualize cross-validation results
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Hyperparameter Tuning: Cross-Validation Results', fontsize=16, fontweight='bold')

# Plot 1: RMSE comparison
param_labels = [f"Config {i+1}\n({r['params']['units1']},{r['params']['units2']})" 
                for i, r in enumerate(tuning_results)]
rmse_means = [r['avg_rmse'] for r in tuning_results]
rmse_stds = [r['std_rmse'] for r in tuning_results]

axes[0].bar(range(len(tuning_results)), rmse_means, yerr=rmse_stds, 
            alpha=0.7, color='steelblue', capsize=5)
axes[0].set_xlabel('Hyperparameter Configuration', fontsize=12, fontweight='bold')
axes[0].set_ylabel('RMSE (Lower is Better)', fontsize=12, fontweight='bold')
axes[0].set_title('Cross-Validation RMSE by Configuration', fontsize=13, fontweight='bold')
axes[0].set_xticks(range(len(tuning_results)))
axes[0].set_xticklabels(param_labels, rotation=45, ha='right')
axes[0].grid(True, alpha=0.3, axis='y')

# Highlight best configuration
best_idx = np.argmin(rmse_means)
axes[0].bar(best_idx, rmse_means[best_idx], color='green', alpha=0.8, label='Best')
axes[0].legend()

# Plot 2: R² comparison
r2_means = [r['avg_r2'] for r in tuning_results]
r2_stds = [r['std_r2'] for r in tuning_results]

axes[1].bar(range(len(tuning_results)), r2_means, yerr=r2_stds, 
            alpha=0.7, color='coral', capsize=5)
axes[1].set_xlabel('Hyperparameter Configuration', fontsize=12, fontweight='bold')
axes[1].set_ylabel('R² Score (Higher is Better)', fontsize=12, fontweight='bold')
axes[1].set_title('Cross-Validation R² by Configuration', fontsize=13, fontweight='bold')
axes[1].set_xticks(range(len(tuning_results)))
axes[1].set_xticklabels(param_labels, rotation=45, ha='right')
axes[1].grid(True, alpha=0.3, axis='y')

# Highlight best configuration
best_idx = np.argmax(r2_means)
axes[1].bar(best_idx, r2_means[best_idx], color='green', alpha=0.8, label='Best')
axes[1].legend()

plt.tight_layout()
plt.show()

print("✅ Cross-validation results visualized!")

### Train Final Model with Best Hyperparameters

Now let's train a model using the best hyperparameters found through cross-validation.

In [ ]:
# Build model with best hyperparameters
print("🏆 Building model with best hyperparameters from cross-validation...")
print("=" * 70)
print(f"Best parameters: {best_params}")

# Build model with best parameters
with tf.device('/CPU:0'):
    best_model = Sequential([
        SimpleRNN(best_params['units1'], activation='relu', 
                 return_sequences=True, input_shape=(sequence_length, n_features)),
        Dropout(best_params['dropout']),
        SimpleRNN(best_params['units2'], activation='relu', return_sequences=False),
        Dropout(best_params['dropout']),
        Dense(16, activation='relu'),
        Dense(1)
    ])
    
    best_model.compile(
        optimizer=Adam(learning_rate=best_params['learning_rate']), 
        loss='mse', 
        metrics=['mae']
    )

print("\n✅ Best model created!")
best_model.summary()

In [ ]:
# Train the best model on full training data
print("🚀 Training best model on full training data...")
print("=" * 70)

# Callbacks
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6)

# Train model
history_best = best_model.fit(
    X_train_seq_scaled, y_train_scaled,
    validation_split=0.2,
    epochs=50,
    batch_size=best_params['batch_size'],
    callbacks=[early_stopping, reduce_lr],
    verbose=1
)

print("\n✅ Best model trained successfully!")

### Compare Tuned Model vs Original Model

Let's compare the performance of the tuned model with the original model.

In [ ]:
# Evaluate best model
y_train_pred_best_scaled = best_model.predict(X_train_seq_scaled, verbose=0)
y_test_pred_best_scaled = best_model.predict(X_test_seq_scaled, verbose=0)

# Inverse transform
y_train_pred_best = target_scaler.inverse_transform(y_train_pred_best_scaled).flatten()
y_test_pred_best = target_scaler.inverse_transform(y_test_pred_best_scaled).flatten()

# Calculate metrics for best model
best_train_rmse = np.sqrt(mean_squared_error(y_train_seq, y_train_pred_best))
best_test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred_best))
best_train_mae = mean_absolute_error(y_train_seq, y_train_pred_best)
best_test_mae = mean_absolute_error(y_test, y_test_pred_best)
best_train_r2 = r2_score(y_train_seq, y_train_pred_best)
best_test_r2 = r2_score(y_test, y_test_pred_best)

# Compare with original model
print("📊 Model Comparison: Original vs Tuned (Best Hyperparameters)")
print("=" * 80)
print(f"\n{'Metric':<15} {'Original Model':<25} {'Tuned Model':<25} {'Improvement':<15}")
print("-" * 80)

# Training metrics
print(f"\n{'TRAINING SET':<15}")
print(f"{'RMSE':<15} {train_rmse:<25.4f} {best_train_rmse:<25.4f} {(train_rmse-best_train_rmse)/train_rmse*100:+.2f}%")
print(f"{'MAE':<15} {train_mae:<25.4f} {best_train_mae:<25.4f} {(train_mae-best_train_mae)/train_mae*100:+.2f}%")
print(f"{'R²':<15} {train_r2:<25.4f} {best_train_r2:<25.4f} {(best_train_r2-train_r2)/abs(train_r2)*100:+.2f}%")

# Test metrics
print(f"\n{'TEST SET':<15}")
print(f"{'RMSE':<15} {test_rmse:<25.4f} {best_test_rmse:<25.4f} {(test_rmse-best_test_rmse)/test_rmse*100:+.2f}%")
print(f"{'MAE':<15} {test_mae:<25.4f} {best_test_mae:<25.4f} {(test_mae-best_test_mae)/test_mae*100:+.2f}%")
print(f"{'R²':<15} {test_r2:<25.4f} {best_test_r2:<25.4f} {(best_test_r2-test_r2)/abs(test_r2)*100:+.2f}%")

print("\n💡 Interpretation:")
if best_test_r2 > test_r2:
    improvement = (best_test_r2 - test_r2) / abs(test_r2) * 100
    print(f"   ✅ Tuned model improved R² by {improvement:.2f}% on test set!")
else:
    print("   ⚠️  Tuned model performance similar to original (may need more tuning)")

### Visualize Comparison: Original vs Tuned Model

Let's create side-by-side visualizations comparing the original and tuned models.

In [ ]:
# Compare predictions visually
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Model Comparison: Original vs Tuned (Best Hyperparameters)', fontsize=16, fontweight='bold')

# Plot 1: Original model - Training
axes[0, 0].scatter(y_train_seq, y_train_pred, alpha=0.5, s=20, color='steelblue')
min_val = min(min(y_train_seq), min(y_train_pred))
max_val = max(max(y_train_seq), max(y_train_pred))
axes[0, 0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect')
axes[0, 0].set_xlabel('Actual RUL', fontsize=11)
axes[0, 0].set_ylabel('Predicted RUL', fontsize=11)
axes[0, 0].set_title(f'Original Model - Train (R² = {train_r2:.4f})', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Tuned model - Training
axes[0, 1].scatter(y_train_seq, y_train_pred_best, alpha=0.5, s=20, color='green')
min_val = min(min(y_train_seq), min(y_train_pred_best))
max_val = max(max(y_train_seq), max(y_train_pred_best))
axes[0, 1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect')
axes[0, 1].set_xlabel('Actual RUL', fontsize=11)
axes[0, 1].set_ylabel('Predicted RUL', fontsize=11)
axes[0, 1].set_title(f'Tuned Model - Train (R² = {best_train_r2:.4f})', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Original model - Test
axes[1, 0].scatter(y_test, y_test_pred, alpha=0.5, s=20, color='coral')
min_val = min(min(y_test), min(y_test_pred))
max_val = max(max(y_test), max(y_test_pred))
axes[1, 0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect')
axes[1, 0].set_xlabel('Actual RUL', fontsize=11)
axes[1, 0].set_ylabel('Predicted RUL', fontsize=11)
axes[1, 0].set_title(f'Original Model - Test (R² = {test_r2:.4f})', fontsize=12, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 4: Tuned model - Test
axes[1, 1].scatter(y_test, y_test_pred_best, alpha=0.5, s=20, color='purple')
min_val = min(min(y_test), min(y_test_pred_best))
max_val = max(max(y_test), max(y_test_pred_best))
axes[1, 1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect')
axes[1, 1].set_xlabel('Actual RUL', fontsize=11)
axes[1, 1].set_ylabel('Predicted RUL', fontsize=11)
axes[1, 1].set_title(f'Tuned Model - Test (R² = {best_test_r2:.4f})', fontsize=12, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("✅ Comparison visualization created!")

### Summary: Benefits of Cross-Validation and Hyperparameter Tuning

**What We Learned:**

1. **K-Fold Cross-Validation**:
   - Tests model on multiple data splits
   - More reliable performance estimate
   - Reduces risk of overfitting to one split

2. **Hyperparameter Tuning**:
   - Systematically tests different parameter combinations
   - Finds optimal settings for your specific data
   - Can significantly improve model performance

3. **Time Series Considerations**:
   - Use TimeSeriesSplit to respect temporal order
   - Prevents data leakage from future to past
   - More realistic evaluation

**Key Takeaways:**
- Cross-validation provides more reliable performance estimates
- Hyperparameter tuning can improve model performance
- Best to tune on cross-validation, then train final model on full data
- Balance between exploration (more combinations) and time (fewer combinations)

**Next Steps:**
- Try more hyperparameter combinations
- Increase number of folds (5-fold or 10-fold)
- Tune more parameters (sequence length, number of layers, etc.)
- Use automated search (GridSearchCV, RandomizedSearchCV)

In [ ]:
# Predictions
y_train_pred_scaled = model.predict(X_train_seq_scaled, verbose=0)
y_test_pred_scaled = model.predict(X_test_seq_scaled, verbose=0)

# Inverse transform
y_train_pred = target_scaler.inverse_transform(y_train_pred_scaled).flatten()
y_test_pred = target_scaler.inverse_transform(y_test_pred_scaled).flatten()

# Calculate metrics
train_rmse = np.sqrt(mean_squared_error(y_train_seq, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_mae = mean_absolute_error(y_train_seq, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)
train_r2 = r2_score(y_train_seq, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("="*60)
print("RNN Model Performance")
print("="*60)
print(f"\nTraining Metrics:")
print(f"  RMSE: {train_rmse:.4f}")
print(f"  MAE:  {train_mae:.4f}")
print(f"  R²:   {train_r2:.4f}")
print(f"\nTest Metrics:")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  MAE:  {test_mae:.4f}")
print(f"  R²:   {test_r2:.4f}")


## 8. Visualizations

### Purpose
Visualize training progress and prediction quality to understand model behavior.

### Visualizations Created:
1. **Training History**: Shows how loss and MAE change during training
   - Decreasing loss = model is learning
   - Validation loss should track training loss (if diverging = overfitting)

2. **Predictions vs Actual**: Scatter plots showing prediction accuracy
   - Points on diagonal = good predictions
   - Spread indicates prediction variance


In [ ]:
# Training history
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('RNN Training History', fontsize=14, fontweight='bold')

axes[0].plot(history.history['loss'], label='Train Loss', linewidth=2)
axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=11)
axes[0].set_ylabel('Loss (MSE)', fontsize=11)
axes[0].set_title('Model Loss', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Train MAE', linewidth=2)
axes[1].plot(history.history['val_mae'], label='Validation MAE', linewidth=2)
axes[1].set_xlabel('Epoch', fontsize=11)
axes[1].set_ylabel('MAE', fontsize=11)
axes[1].set_title('Model MAE', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


In [ ]:
# Predictions vs Actual
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('RNN Predictions vs Actual', fontsize=14, fontweight='bold')

# Training set
axes[0].scatter(y_train_seq, y_train_pred, alpha=0.5, s=20)
min_val = min(min(y_train_seq), min(y_train_pred))
max_val = max(max(y_train_seq), max(y_train_pred))
axes[0].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual RUL', fontsize=11)
axes[0].set_ylabel('Predicted RUL', fontsize=11)
axes[0].set_title(f'Train Set (R² = {train_r2:.4f})', fontsize=12, fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Test set
axes[1].scatter(y_test, y_test_pred, alpha=0.5, s=20)
min_val = min(min(y_test), min(y_test_pred))
max_val = max(max(y_test), max(y_test_pred))
axes[1].plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual RUL', fontsize=11)
axes[1].set_ylabel('Predicted RUL', fontsize=11)
axes[1].set_title(f'Test Set (R² = {test_r2:.4f})', fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
